### INITIALIZATION

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"

In [ ]:
import math
import random

In [ ]:
from gensim.models import Word2Vec

In [ ]:
from sklearn.metrics import f1_score

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, BatchSampler, Sampler
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.amp import autocast, GradScaler

torch._inductor.config.triton.cudagraph_skip_dynamic_graphs=True

torch.backends.fp32_precision = "tf32"
torch.backends.cudnn.fp32_precision = "tf32"
torch.backends.cudnn.rnn.fp32_precision = "tf32"

In [ ]:
from time import perf_counter

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
assert torch.cuda.is_available()
device = torch.device("cuda")
print(f"Device: {device}")

### DATASET

In [ ]:
def convert_labels(label):
    if label in {"fake", "hate", "rumor", "unreliable", "conspiracy", "bias", "junksci", "satire"}:
        return 0
    elif label in {"reliable", "political", "clickbait"}:
        return 1
    else:
        return pd.NA

In [ ]:
metadata = pd.read_csv("./../data/995,000_rows.csv", usecols=["type"])

In [ ]:
converted_labels = metadata["type"].apply(convert_labels)
converted_labels.head()

In [ ]:
del metadata

#### WORD EMBEDDING MODEL

In [ ]:
model1 = Word2Vec.load("model1.model")

In [ ]:
model1.wv["hello"]

In [ ]:
print(f"Embedding size: {len(model1.wv["hello"])}")

In [ ]:
model1.wv.get_index("hello")

In [ ]:
def tokenlist_to_tokenindexes(tokens):
    return torch.tensor([model1.wv.get_index(token) for token in tokens if token in model1.wv], dtype=torch.long)

In [ ]:
file_chunks = pd.read_csv("./../data/data_stemmed.csv", index_col=0, chunksize=50000) 

In [ ]:
temp_chunks = []

In [ ]:
for chunk in file_chunks:
    chunk = chunk["content"].apply(lambda s: s.split(" "))
    chunk = chunk.apply(tokenlist_to_tokenindexes)
    temp_chunks.append(chunk)


In [ ]:
data = pd.concat(temp_chunks).to_frame()
data.head()

In [ ]:
data.memory_usage(deep=True)

In [ ]:
data["type"] = converted_labels
data.head()

In [ ]:
print(data.isna().sum())
print(data.shape)

In [ ]:
data.dropna(inplace=True)
print(data.isna().sum())
print(data.shape)

In [ ]:
# Shuffle dataset randomly
data = data.sample(frac=1, random_state=0)

In [ ]:
data.reset_index(inplace=True, drop=True)

In [ ]:
# GROUP BY BUCKET SIZE
buckets = [0, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768]
data["cuts"] = pd.cut(data["content"].apply(lambda x: x.shape[0]), bins=buckets, labels=buckets[1:])

In [ ]:
for bucket_size, group in data.groupby("cuts", observed=True):
    print(f"Length: {len(group)} Bucketsize: {bucket_size}")

In [ ]:
split_index1 = int(len(data)*0.8)
split_index2 = int(len(data)*0.9)

In [ ]:
train_set = data.iloc[:split_index1]
val_set = data.iloc[split_index1:split_index2]
test_set = data.iloc[split_index2:]

In [ ]:
print(len(train_set))
print(len(val_set))
print(len(test_set))

In [ ]:
class MySampler(Sampler):
    def __init__(self, groups, bucket_to_batch_size):
        self.groups = groups
        self.bucket_to_batch_size = bucket_to_batch_size

    def __iter__(self):
        batches = []
        
        for bucket_size, group_indices in self.groups.indices.items():    
            indices = list(group_indices)
            random.shuffle(indices)

            new_batch_size = self.bucket_to_batch_size[bucket_size]
            for i in range(0, len(indices), new_batch_size):
                batch = indices[i:i+new_batch_size]
                if batch:
                    batches.append(batch)

        random.shuffle(batches)
        
        yield from batches

    def __len__(self):
        return len(self.batches)

In [ ]:
class ArticleDataset(Dataset):
    def __init__(self, df):
        self.data = df

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        articles = self.data["content"].iloc[idx]
        padded_articles = pad_sequence(articles.tolist(), batch_first=True)
        lengths = torch.tensor([a.size(0) for a in articles], dtype=torch.long).cpu()
        
        labels = self.data["type"].iloc[idx]
        labels_tensor = torch.from_numpy(labels.to_numpy(dtype=np.float32))

        # article_tensor = self.embedding(article)
        # label_tensor = torch.tensor([label], dtype=torch.float32)
        return padded_articles, lengths, labels_tensor

#### TRAIN, VALIDATION, TEST DATASETS

In [ ]:
bucket_to_batch_size = {
    8: 512, 
    16: 512, 
    32: 512, 
    64: 512, 
    128: 512, 
    256: 512, 
    512: 256, 
    1024: 128, 
    2048: 64, 
    4096: 32, 
    8192: 16, 
    16384: 8, 
    32768: 4,
}

In [ ]:
train_sampler = MySampler(train_set.groupby("cuts", observed=True), bucket_to_batch_size)

train_dataloader = DataLoader(
    ArticleDataset(train_set), 
    batch_size=1,
    sampler=train_sampler,
    num_workers=4,
    collate_fn=None, 
    pin_memory=True,
    persistent_workers=True,
)

In [ ]:
# buckettosize = {8: 32, 16: 32, 32: 32, 64: 32, 128: 32, 256: 32, 512: 16, 1024: 8, 2048: 4, 4096: 2, 8192: 1, 16384: 1, 32768: 1}

In [ ]:
val_sampler = MySampler(val_set.groupby("cuts", observed=True), bucket_to_batch_size)

val_dataloader = DataLoader(
    ArticleDataset(val_set), 
    batch_size=1,
    sampler=val_sampler,
    num_workers=4,
    collate_fn=None, 
    pin_memory=True,
    persistent_workers=True,
)

In [ ]:
test_sampler = MySampler(test_set.groupby("cuts", observed=True), bucket_to_batch_size)

test_dataloader = DataLoader(
    ArticleDataset(test_set), 
    batch_size=1,
    sampler=val_sampler,
    num_workers=4,
    collate_fn=None, 
    pin_memory=True,
    persistent_workers=True,
)

### NN MODEL

In [ ]:
class WordRNN(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, num_layers, tagset_size=1):
        super(WordRNN, self).__init__()

        self.word_embeddings = nn.Embedding.from_pretrained(torch.tensor(model1.wv.vectors))
        self.word_embeddings.weight.requires_grad = False
        
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        
    def forward(self, article_data, article_lengths):
        article_embeddings_tensor = self.word_embeddings(article_data)
        
        packed_data = pack_padded_sequence(article_embeddings_tensor, article_lengths, batch_first=True, enforce_sorted=False)
        output, (h_n, c_n) = self.lstm(article_embeddings_tensor)
        last_hidden = h_n[-1]
        
        logit = self.fc(last_hidden)

        return logit

In [ ]:
model = WordRNN(64, 128, 2)
# Embedding dim: 64
# Hidden dim: 128
# Number of LSTM layers: 2

model = model.to("cuda")
model = torch.compile(model, mode="max-autotune-no-cudagraphs")
model

### TRAINING

In [ ]:
scaler = GradScaler()

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
def train_one_batch(model, batch_articles, article_lengths, batch_labels):
    batch_articles = batch_articles.squeeze(0)
    article_lengths = article_lengths.squeeze(0)
    batch_labels = batch_labels.squeeze(0)

    optimizer.zero_grad(set_to_none=True)

    batch_articles = batch_articles.to(device, non_blocking=True)
    batch_labels = batch_labels.to(device, non_blocking=True)

    with autocast(device_type="cuda"):
        outputs = model(batch_articles, article_lengths).view(batch_labels.size(0))
        batch_loss = criterion(outputs, batch_labels)
    
    scaler.scale(loss).backward()
    # loss.backward()
        
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    scaler.step(optimizer)
    # optimizer.step()
        
    scaler.update()

    return batch_loss.item()

In [ ]:
def evaluate_one_batch(model, batch_articles, article_lengths, batch_labels):
    batch_articles = batch_articles.squeeze(0)
    article_lengths = article_lengths.squeeze(0)
    batch_labels = batch_labels.squeeze(0)
    
    batch_articles = batch_articles.to(device, non_blocking=True)
    batch_labels = batch_labels.to(device, non_blocking=True)
    
    outputs = model(batch_articles, article_lengths).view(batch_labels.size(0))
    batch_loss = criterion(outputs, batch_labels)

    return batch_loss.item()

In [ ]:
epoch_losses = []

In [ ]:
start_time = perf_counter()
last_eval_avg_loss = np.inf
epochs_since_last_improvement = 0

for epoch in range(0, 5):
    model.train()
    epoch_training_loss = 0.0
    for index, (batch_articles, article_lengths, batch_labels) in enumerate(train_dataloader):
        batch_loss = train_one_batch(model, batch_articles, article_lengths, batch_labels)
        epoch_total_loss += batch_loss
            
    avg_epoch_training_loss = epoch_training_loss / (index + 1)

    
    model.eval()
    epoch_eval_loss = 0
    with torch.no_grad():
        for index, (batch_articles, article_lengths, batch_labels) in enumerate(val_dataloader):
            batch_loss = evaluate_one_batch(model, batch_articles, article_lengths, batch_labels)            
            epoch_eval_loss += batch_loss

    avg_epoch_eval_loss = eval_total_loss / (index + 1)

    
    end_time = perf_counter()
    print(f"\nEpoch {epoch} avg training loss: {avg_epoch_training_loss}.")
    print(f"Epoch {epoch} avg eval loss: {avg_epoch_eval_loss}. Changed {eval_avg_loss - last_eval_loss}")
    print(f"Epoch {epoch} elapsed time: {end_time - start_time} seconds.\n")

    if avg_epoch_eval_loss < last_eval_avg_loss:
        epochs_since_last_improvement = 0
    else:
        epochs_since_last_improvement += 1
        print("No improvement")
        if epochs_since_last_improvement == 2:
            print(f"Stopping early at epoch {epoch}")
            break
            
    last_eval_avg_loss = avg_epoch_eval_loss
    epoch_losses.append(avg_epoch_eval_loss)
    start_time = end_time

In [ ]:
torch.save(model.state_dict(), "model2_25epoch.pth")

#### TESTING

In [ ]:
model.load_state_dict(torch.load("model2_25epoch.pth"))

In [ ]:
torch.cuda.empty_cache()

In [ ]:
model.eval()
predictions = []
true_labels = []

with torch.no_grad():
    for index, (batch_articles, article_lengths, batch_labels) in enumerate(test_dataloader):
        batch_articles = batch_articles.squeeze(0)
        article_lengths = article_lengths.squeeze(0)
        batch_labels = batch_labels.squeeze(0)

        batch_articles = batch_articles.to(device, non_blocking=True)

        output = model(batch_articles, article_lengths)

        probs = torch.sigmoid(output)
        prediction = (probs > 0.5).long().cpu()
        
        predictions.append(prediction)
        true_labels.append(batch_labels)

In [ ]:
test_score = f1_score(torch.cat(true_labels), torch.cat(predictions))
test_score